In [ ]:

import os
from pathlib import Path
import kagglehub
import shutil


kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
access_token_file = kaggle_dir / "access_token"

token_val = "KGAT_3756b002f905ff5b2bd6d91a033bf23d"
with open(access_token_file, "w", encoding="utf-8") as f:
    f.write(token_val)
print(f"Kaggle API Token configured at: {access_token_file}")


os.environ["KAGGLE_API_TOKEN"] = token_val


print("Downloading dataset (dwsonder) via kagglehub...")
path = kagglehub.dataset_download('nderalparslan/dwsonder')
print(f"Dataset downloaded successfully to: {path}")


local_dataset_path = Path('./dwsonder')
if local_dataset_path.exists():
    shutil.rmtree(local_dataset_path)
shutil.copytree(path, local_dataset_path)
print(f"Copied dataset locally to: {local_dataset_path.resolve()}")

c:\Users\ROG\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kaggle API Token configured at: C:\Users\ROG\.kaggle\access_token


100%|██████████| 2.35G/2.35G [11:59<00:00, 3.51MB/s] 

Extracting model files...


Dataset downloaded successfully to: C:\Users\ROG\.cache\kagglehub\datasets\nderalparslan\dwsonder\versions\1
Copied dataset locally to: C:\Users\ROG\.gemini\antigravity\scratch\dws_depth_detection\dwsonder


In [ ]:
!pip install opencv-python ultralytics torch torchvision timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 921.5/921.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
import cv2
import torch
import numpy as np
from torchvision.transforms import Compose, Resize, Normalize, ToTensor
from ultralytics import YOLO

# Load MiDaS model
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)  # Use MiDaS_small for faster inference
midas.eval()

# MiDaS transforms
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
transform = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True).small_transform

# Load YOLOv8 for object detection
yolo_model = YOLO("yolov8n.pt")  # Use yolov8n for lightweight detection

# Video input and output
input_video = "videoplayback.mp4"  # Input video name
output_video = "output_videoplayback_with_distances.mp4"  # Output video name
cap = cv2.VideoCapture(input_video)

# Set up video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Alternative: 'XVID'
out = cv2.VideoWriter(output_video, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

# Function to normalize depth maps
def normalize_depth(depth_map):
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
    depth_map = (depth_map * 255).astype(np.uint8)
    return depth_map

frame_count = 0  # Debug frame count

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Step 1: Object Detection with YOLO
    detections = yolo_model.predict(source=frame, conf=0.5)
    result = detections[0]

    # Step 2: Depth Estimation with MiDaS
    input_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = transform(input_image).to(device)
    with torch.no_grad():
        depth_map = midas(input_batch)
        depth_map = depth_map.squeeze().cpu().numpy()

    # Resize depth_map to match the original frame size
    depth_map_resized = cv2.resize(depth_map, (frame.shape[1], frame.shape[0]))

    # Step 3: Annotate the frame with depth + object bounding boxes
    annotated_frame = frame.copy()
    for det in result.boxes.data.tolist():
        x1, y1, x2, y2, conf, cls = map(int, det)
        label = result.names[cls]

        # Extract depth value for the center of the bounding box
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        depth_value = depth_map_resized[center_y, center_x]

        # Annotate the object with depth in meters
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated_frame,
            f"{label}: {depth_value:.2f} m",  # Displaying depth in meters
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )

    # Write frame to output video
    out.write(annotated_frame)

cap.release()
out.release()

print(f"Processed video saved as {output_video}")
print(f"Total frames processed: {frame_count}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
Loading weights:  None
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:00<00:00, 256MB/s]


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Processed video saved as output_videoplayback_with_distances.mp4
Total frames processed: 0


In [ ]:
import cv2
import torch
import numpy as np
from torchvision.transforms import Compose, Resize, Normalize, ToTensor
from ultralytics import YOLO

# Load MiDaS model
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)  # Use MiDaS_small for faster inference
midas.eval()

# MiDaS transforms
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
transform = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True).small_transform

# Load YOLOv8 for object detection
yolo_model = YOLO("yolov8n.pt")  # Use yolov8n for lightweight detection

# Video input and output
input_video = "WhatsApp Video 2026-06-10 at 2.35.34 PM.mp4"  # Input video name
output_video = "output_videoplayback_with_distances_cm2.mp4"  # Output video name
cap = cv2.VideoCapture(input_video)

# Set up video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Alternative: 'XVID'
out = cv2.VideoWriter(output_video, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

# Function to normalize depth maps
def normalize_depth(depth_map):
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
    depth_map = (depth_map * 255).astype(np.uint8)
    return depth_map

frame_count = 0  # Debug frame count

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Step 1: Object Detection with YOLO
    detections = yolo_model.predict(source=frame, conf=0.5)
    result = detections[0]

    # Step 2: Depth Estimation with MiDaS
    input_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = transform(input_image).to(device)
    with torch.no_grad():
        depth_map = midas(input_batch)
        depth_map = depth_map.squeeze().cpu().numpy()

    # Resize depth_map to match the original frame size
    depth_map_resized = cv2.resize(depth_map, (frame.shape[1], frame.shape[0]))

    # Step 3: Annotate the frame with depth + object bounding boxes
    annotated_frame = frame.copy()
    for det in result.boxes.data.tolist():
        x1, y1, x2, y2, conf, cls = map(int, det)
        label = result.names[cls]

        # Extract depth value for the center of the bounding box
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        depth_value = depth_map_resized[center_y, center_x]

        # Convert depth to centimeters
        depth_in_cm = depth_value  # Scale depth from meters to centimeters

        # Annotate the object with depth in centimeters
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated_frame,
            f"{label}: {depth_in_cm:.1f} cm",  # Displaying depth in centimeters
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )

    # Write frame to output video
    out.write(annotated_frame)

cap.release()
out.release()

print(f"Processed video saved as {output_video}")
print(f"Total frames processed: {frame_count}")


Using cache found in C:\Users\ROG/.cache\torch\hub\intel-isl_MiDaS_master
c:\Users\ROG\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:  None


Using cache found in C:\Users\ROG/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\ROG/.cache\torch\hub\intel-isl_MiDaS_master


Processed video saved as output_videoplayback_with_distances_cm2.mp4
Total frames processed: 0


In [ ]:
!mkdir -p ~/.kaggle
!cp path_to_your_kaggle_json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nderalparslan/dwsonder
!unzip dwsonder.zip -d dwsonder


Streaming output truncated to the last 5000 lines.
  inflating: dwsonder/images/1cc00d946d527251.txt  
  inflating: dwsonder/images/1cc919e1ed5a71e1.jpg  
  inflating: dwsonder/images/1cc919e1ed5a71e1.txt  
  inflating: dwsonder/images/1ccba45e8e4491a3.jpg  
  inflating: dwsonder/images/1ccba45e8e4491a3.txt  
  inflating: dwsonder/images/1cdb00bbeb6f8852.jpg  
  inflating: dwsonder/images/1cdb00bbeb6f8852.txt  
  inflating: dwsonder/images/1cdd6c9709932813.jpg  
  inflating: dwsonder/images/1cdd6c9709932813.txt  
  inflating: dwsonder/images/1d01e2a7bebbb07f.jpg  
  inflating: dwsonder/images/1d01e2a7bebbb07f.txt  
  inflating: dwsonder/images/1d05f63fa8d0b663.jpg  
  inflating: dwsonder/images/1d05f63fa8d0b663.txt  
  inflating: dwsonder/images/1d0e6bb123befc16.jpg  
  inflating: dwsonder/images/1d0e6bb123befc16.txt  
  inflating: dwsonder/images/1d3362e5d8346e5b.jpg  
  inflating: dwsonder/images/1d3362e5d8346e5b.txt  
  inflating: dwsonder/images/1d624eb29f35af59.jpg  
  inflating: 

In [2]:
import shutil
import random
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)

# Define the root dataset directory
dataset_path = Path('./dwsonder')

# The directory containing all images and labels
source_dir = dataset_path / 'images'

# Verify the source directory exists
if not source_dir.exists():
    raise FileNotFoundError(f"The source directory '{source_dir}' does not exist.")

# Create the destination directories for the split
train_images_path = dataset_path / 'images' / 'train'
val_images_path = dataset_path / 'images' / 'val'
train_labels_path = dataset_path / 'labels' / 'train'
val_labels_path = dataset_path / 'labels' / 'val'

for directory in [train_images_path, val_images_path, train_labels_path, val_labels_path]:
    directory.mkdir(parents=True, exist_ok=True)

# Define the image extension we are interested in
IMAGE_EXTENSION = '.jpg'

# Get a list of all image files
image_files = [f for f in source_dir.iterdir() if f.is_file() and f.suffix.lower() == IMAGE_EXTENSION]

print(f"Total images found: {len(image_files)}")

# Identify which images do not have corresponding labels
missing_labels = []
for image in image_files:
    label_file = source_dir / f"{image.stem}.txt"
    if not label_file.exists():
        missing_labels.append(image.name)

if missing_labels:
    print(f"Warning: {len(missing_labels)} images do not have corresponding label files and will be skipped.")
    # Optionally, write missing labels to a file
    missing_labels_file = dataset_path / 'missing_labels.txt'
    with missing_labels_file.open('w') as f:
        for name in missing_labels:
            f.write(f"{name}\n")
    print(f"List of images without labels saved to '{missing_labels_file}'.")
else:
    print("All images have corresponding label files.")

# Filter out images without labels
image_files = [f for f in image_files if (source_dir / f"{f.stem}.txt").exists()]

print(f"Images with labels: {len(image_files)}")

# If no images remain, exit
if not image_files:
    print("No images with corresponding labels found. Exiting.")
    exit()

# Shuffle images
random.shuffle(image_files)

# Split into train (80%) and val (20%)
train_count = int(0.8 * len(image_files))
train_files = image_files[:train_count]
val_files = image_files[train_count:]

print(f"Train set size: {len(train_files)} images")
print(f"Validation set size: {len(val_files)} images")

# Function to move image and label files
def move_files(file_list, dest_images, dest_labels):
    for image in file_list:
        # Define label file
        label_file = source_dir / f"{image.stem}.txt"

        # Define destination paths
        dest_image = dest_images / image.name
        dest_label = dest_labels / label_file.name

        # Move image
        try:
            shutil.move(str(image), str(dest_image))
        except shutil.Error as e:
            print(f"Error moving image '{image.name}': {e}")

        # Move label
        try:
            shutil.move(str(label_file), str(dest_label))
        except shutil.Error as e:
            print(f"Error moving label '{label_file.name}': {e}")

# Move the training files
print("Moving training files...")
move_files(train_files, train_images_path, train_labels_path)

# Move the validation files
print("Moving validation files...")
move_files(val_files, val_images_path, val_labels_path)

print(f"Dataset reorganized: {len(train_files)} train images, {len(val_files)} val images.")
print(f"Training images are in '{train_images_path}'")
print(f"Training labels are in '{train_labels_path}'")
print(f"Validation images are in '{val_images_path}'")
print(f"Validation labels are in '{val_labels_path}'")


Total images found: 5039
All images have corresponding label files.
Images with labels: 5039
Train set size: 4031 images
Validation set size: 1008 images
Moving training files...
Moving validation files...
Dataset reorganized: 4031 train images, 1008 val images.
Training images are in 'dwsonder\images\train'
Training labels are in 'dwsonder\labels\train'
Validation images are in 'dwsonder\images\val'
Validation labels are in 'dwsonder\labels\val'


In [3]:
# Create YAML configuration file
yaml_content = """
path: ./dwsonder  # Change this to your dataset path
train: images/train  # Training images directory
val: images/val      # Validation images directory

names:
  0: door   # Class 0 - Door
  1: stairs # Class 1 - Stairs
  2: window # Class 2 - Window
"""

# Write the content to a new YAML file
with open('./dwsonder/dwsonder.yaml', 'w') as f:
    f.write(yaml_content)

print("YAML file created!")


YAML file created!


In [4]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
print(model.model)

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
  

In [1]:
from ultralytics import YOLO

# Load our best trained model weights from the first 5 epochs
model = YOLO("runs/detect/train/weights/best.pt")

# Fine-tune for 5 more epochs
model.train(data="./dwsonder/dwsonder.yaml", epochs=5, imgsz=640,freeze=10,batch=8)



New https://pypi.org/project/ultralytics/8.4.61 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dwsonder/dwsonder.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000015C4A68FDC0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # load YOLOv8n (pretrained on COCO)
print(model.names)

In [1]:
import cv2
import torch
import numpy as np
from torchvision.transforms import Compose, Resize, Normalize, ToTensor
from ultralytics import YOLO
from pathlib import Path
from narrator import AudioNarrator

# Load MiDaS model
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)  # Use MiDaS_small for faster inference
midas.eval()

# MiDaS transforms
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
transform = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True).small_transform

# 1. Load the COCO model for general objects (chairs, bottles, etc.)
print("Loading COCO YOLO model for general objects...")
coco_model = YOLO("yolov8n.pt")

# 2. Load the custom DWS model for doors, windows, stairs
yolo_model_path = "yolov8n.pt"  # default fallback
detect_dir = Path("runs/detect")
if detect_dir.exists():
    best_files = list(detect_dir.glob("**/weights/best.pt"))
    if best_files:
        # Sort by modification time to get the absolute newest weights
        best_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        yolo_model_path = str(best_files[0])

print(f"Loading custom DWS YOLO model from: {yolo_model_path}")
custom_model = YOLO(yolo_model_path)

# Initialize the non-blocking background audio narrator
narrator = AudioNarrator()
narrator.speak("System initialized. Dual object detection and depth mapping activated.")

# Video input and output
input_video = "./WhatsApp Video 2026-06-08 at 1.55.49 AM.mp4"  # Input video name
output_video = "output_videoplayback_with_distances_cm.mp4"  # Output video name
cap = cv2.VideoCapture(input_video)

# Set up video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Alternative: 'XVID'
out = cv2.VideoWriter(output_video, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

# Function to normalize depth maps
def normalize_depth(depth_map):
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
    depth_map = (depth_map * 255).astype(np.uint8)
    return depth_map

frame_count = 0  # Debug frame count

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Step 1: Run predictions with both models
    coco_results = coco_model.predict(source=frame, conf=0.5, verbose=False)[0]
    custom_results = custom_model.predict(source=frame, conf=0.3, verbose=False)[0]

    # Step 2: Depth Estimation with MiDaS
    input_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = transform(input_image).to(device)
    with torch.no_grad():
        depth_map = midas(input_batch)
        depth_map = depth_map.squeeze().cpu().numpy()
 
    # Resize depth_map to match the original frame size
    depth_map_resized = cv2.resize(depth_map, (frame.shape[1], frame.shape[0]))

    annotated_frame = frame.copy()

    # Step 3: Draw boxes from both models
    # Color coding: Blue (255, 0, 0) for COCO general objects, Green (0, 255, 0) for DWS structural elements
    all_detections = [
        (coco_results, 0.5, (255, 0, 0)),      # COCO general detections
        (custom_results, 0.3, (0, 255, 0))     # DWS custom detections
    ]

    frame_detections = []  # To hold all detections on this frame for audio processing

    for result, conf_thresh, color in all_detections:
        for det in result.boxes.data.tolist():
            x1, y1, x2, y2, conf, cls = det
            if conf < conf_thresh:
                continue
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            label = result.names[int(cls)]
            frame_detections.append((x1, y1, x2, y2, conf, label))

            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2

            # Clamp center coordinates to be within frame bounds
            center_x = min(max(center_x, 0), frame.shape[1] - 1)
            center_y = min(max(center_y, 0), frame.shape[0] - 1)

            depth_value = depth_map_resized[center_y, center_x]
            depth_in_cm = depth_value

            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                annotated_frame,
                f"{label}: {depth_in_cm:.1f} cm",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2,
            )

    # Trigger audio announcements in the background thread
    narrator.announce_objects(frame_detections, depth_map_resized)

    # Write frame to output video
    out.write(annotated_frame)

cap.release()
out.release()
narrator.stop()  # Clean up and stop the background thread

print(f"Processed video saved as {output_video}")
print(f"Total frames processed: {frame_count}")


Using cache found in C:\Users\ROG/.cache\torch\hub\intel-isl_MiDaS_master
c:\Users\ROG\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:  None


Using cache found in C:\Users\ROG/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\ROG/.cache\torch\hub\intel-isl_MiDaS_master


Loading COCO YOLO model for general objects...
Loading custom DWS YOLO model from: runs\detect\train-3\weights\best.pt
Processed video saved as output_videoplayback_with_distances_cm.mp4
Total frames processed: 136
